# U6-01 · SQLite, consultas SQL y documentos JSON

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 6*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894
> Datos: Medellín, R. y Serna, J. (2012). *Restaurant & consumer data*. UCI, CC BY 4.0. https://doi.org/10.24432/C5DP41

## 6.1 · Bases de datos relacionales

`t1_sqlite.py`

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("bicis.db")         # un archivo, sin servidor
h = pd.read_csv("hour.csv")
h[["dteday", "season", "holiday", "workingday"]].drop_duplicates(
    "dteday").to_sql("dias", con, if_exists="replace", index=False)
h[["dteday", "hr", "weathersit", "temp", "hum", "cnt"]].to_sql(
    "horas", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS i_fecha ON horas(dteday)")
print(con.execute("SELECT COUNT(*) FROM dias").fetchone(),
      con.execute("SELECT COUNT(*) FROM horas").fetchone())
consulta = """
SELECT d.workingday AS laborable, h.hr AS hora,
       ROUND(AVG(h.cnt), 1) AS media
FROM horas h JOIN dias d ON h.dteday = d.dteday
WHERE h.hr IN (8, 17)
GROUP BY d.workingday, h.hr
ORDER BY d.workingday, h.hr"""
print(pd.read_sql(consulta, con).to_string(index=False))

## 6.2 · Del almacén al modelo y de vuelta

`t2_entrenar.py`

In [ ]:
import sqlite3
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

con = sqlite3.connect("bicis.db")
q = """SELECT h.hr, d.workingday, h.temp, h.hum, h.weathersit,
              h.cnt, h.dteday
       FROM horas h JOIN dias d USING (dteday)"""
df = pd.read_sql(q, con)
ent = df.dteday < "2012-07-01"
X = df.drop(columns=["cnt", "dteday"])
m = HistGradientBoostingRegressor(loss="poisson", random_state=0)
m.fit(X[ent], df.cnt[ent])
df["prevision"] = m.predict(X).round(1)
df[["dteday", "hr", "prevision"]].to_sql(
    "previsiones", con, if_exists="replace", index=False)
print(pd.read_sql("""SELECT p.dteday, p.hr, h.cnt, p.prevision
    FROM previsiones p JOIN horas h USING (dteday, hr)
    WHERE p.dteday = '2012-12-24' AND p.hr BETWEEN 15 AND 18""",
    con).to_string(index=False))

## 6.3 · Bases de datos no relacionales

`t3_documentos.py`

In [ ]:
import json
import pandas as pd

g = pd.read_csv("geoplaces2.csv")
coc = pd.read_csv("chefmozcuisine.csv").groupby("placeID") \
    .Rcuisine.agg(list)
val = pd.read_csv("rating_final.csv").groupby("placeID")
docs = []
for _, f in g.iterrows():                 # un documento por local
    i = f.placeID
    docs.append({"_id": int(i), "nombre": f["name"],
                 "ciudad": f.city, "precio": f.price,
                 "cocinas": coc.get(i, []),
                 "valoraciones": [{"usuario": u, "nota": int(n)}
                                  for u, n in zip(
                                      val.get_group(i).userID,
                                      val.get_group(i).rating)]})
with open("restaurantes.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False)
d0 = dict(docs[0], valoraciones=docs[0]["valoraciones"][:2])
print(json.dumps(d0, ensure_ascii=False))
print("valoraciones del primer local:", len(docs[0]["valoraciones"]))
pizza = [d["nombre"] for d in docs if "Pizzeria" in d["cocinas"]]
print("pizzerías:", len(pizza), pizza[:3])

**Prueba.** Escribe una consulta SQL que devuelva el día con más alquileres de cada año.

**Ejercicio.** Busca en los documentos JSON los restaurantes con más de diez valoraciones.